# Local ONNX resume refiner for Next.js

This notebook fine-tunes a Hugging Face sequence-to-sequence model with the MIT-licensed Hugging Face `ssbML/resumes` dataset, then exports a local ONNX package for the Next.js app. No Hugging Face inference API is used at runtime. The application passes only already-parsed skills, experience, and projects to this model.

The selected dataset is MIT-licensed according to its Hub card. Review both the dataset and base-model license before publishing the resulting model.

In [6]:
%pip install -qU transformers datasets accelerate sentencepiece optimum-onnx onnxruntime evaluate rouge_score


Note: you may need to restart the kernel to use updated packages.


In [7]:
from pathlib import Path
import json
import shutil
from datasets import load_dataset
from transformers import (
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
)

DATASET_ID = "ssbML/resumes"
BASE_MODEL = "google/flan-t5-small"
MAX_TRAIN_ROWS = 1500  # Increase only when training with sufficient GPU time.
MODEL_DIR = Path("artifacts/resume-refiner-pytorch")
ONNX_DIR = Path("public/models/resume-refiner")


In [8]:
dataset = load_dataset(DATASET_ID, split="train")
dataset = dataset.shuffle(seed=42).select(range(min(MAX_TRAIN_ROWS, len(dataset))))
dataset.column_names


C:\Users\USER\AppData\Roaming\Python\Python314\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\USER\.cache\huggingface\hub\datasets--ssbML--resumes. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Generating train split: 100%|██████████| 4817/4817 [00:02<00:00, 2144.16 examples/s]


['personal_info',
 'experience',
 'education',
 'skills',
 'projects',
 'certifications',
 'internships',
 'workshops',
 'publications',
 'achievements',
 'teaching_experience']

In [9]:
def as_text(value):
    if value is None:
        return ""
    if isinstance(value, (list, dict)):
        return json.dumps(value, ensure_ascii=False)
    return str(value)

def parsed_fields(row):
    # The training input has the same three fields the Next.js parser sends at runtime.
    return {
        "skills": as_text(row.get("skills"))[:450],
        "experience": as_text(row.get("experience"))[:950],
        "projects": as_text(row.get("projects"))[:750],
    }

def concise_target(fields):
    return (
        f"Skills: {fields['skills'][:260]}\n"
        f"Experience: {fields['experience'][:520]}\n"
        f"Projects: {fields['projects'][:420]}"
    )

def to_training_record(row):
    fields = parsed_fields(row)
    return {
        "input_text": "Summarize these parsed resume fields concisely: " + json.dumps(fields),
        "target_text": concise_target(fields),
    }

records = dataset.map(to_training_record, remove_columns=dataset.column_names)
split = records.train_test_split(test_size=0.1, seed=42)
split


Map:   0%|          | 0/1500 [00:00<?, ? examples/s]

Map: 100%|██████████| 1500/1500 [00:02<00:00, 528.07 examples/s]


DatasetDict({
    train: Dataset({
        features: ['input_text', 'target_text'],
        num_rows: 1350
    })
    test: Dataset({
        features: ['input_text', 'target_text'],
        num_rows: 150
    })
})

In [10]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
model = AutoModelForSeq2SeqLM.from_pretrained(BASE_MODEL)

def tokenize(batch):
    inputs = tokenizer(batch["input_text"], max_length=512, truncation=True)
    labels = tokenizer(text_target=batch["target_text"], max_length=256, truncation=True)
    inputs["labels"] = labels["input_ids"]
    return inputs

tokenized = split.map(tokenize, batched=True, remove_columns=split["train"].column_names)
training_args = Seq2SeqTrainingArguments(
    output_dir=str(MODEL_DIR),
    learning_rate=3e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=1,
    weight_decay=0.01,
    predict_with_generate=True,
    logging_steps=25,
    save_strategy="no",
    report_to=[],
)
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["test"],
    data_collator=DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model),
)
trainer.train()
trainer.save_model(str(MODEL_DIR))
tokenizer.save_pretrained(str(MODEL_DIR))


Map: 100%|██████████| 150/150 [00:00<00:00, 326.14 examples/s]


Step,Training Loss
25,0.828500
50,0.464000
75,0.347200
100,0.300100
125,0.258500
150,0.246100
175,0.223100
200,0.218900
225,0.189800
250,0.195800


('artifacts\\resume-refiner-pytorch\\tokenizer_config.json',
 'artifacts\\resume-refiner-pytorch\\special_tokens_map.json',
 'artifacts\\resume-refiner-pytorch\\spiece.model',
 'artifacts\\resume-refiner-pytorch\\added_tokens.json',
 'artifacts\\resume-refiner-pytorch\\tokenizer.json')

In [3]:
from optimum.exporters.onnx import main_export
from optimum.exporters.base import ExporterConfig
from pathlib import Path
import shutil

MODEL_DIR = Path("artifacts/resume-refiner-pytorch")
ONNX_DIR = Path("public/models/resume-refiner")

# Workaround for Python 3.14+ where functools.partial is a descriptor
def patched_init(
    self,
    config,
    task: str,
    int_dtype: str = "int64",
    float_dtype: str = "fp32",
):
    self.task = task
    self._config = config
    self._normalized_config = self.__class__.NORMALIZED_CONFIG_CLASS(self._config)
    self.int_dtype = int_dtype
    self.float_dtype = float_dtype

ExporterConfig.__init__ = patched_init

if ONNX_DIR.exists():
    shutil.rmtree(ONNX_DIR)
ONNX_DIR.mkdir(parents=True)
main_export(
    str(MODEL_DIR),
    output=ONNX_DIR / "onnx",
    task="text2text-generation-with-past",
)

# Transformers.js needs the model configuration and tokenizer beside the onnx/ directory.
for source in MODEL_DIR.iterdir():
    if source.is_file() and source.suffix in {".json", ".model"}:
        shutil.copy2(source, ONNX_DIR / source.name)

sorted(path.relative_to(ONNX_DIR).as_posix() for path in ONNX_DIR.rglob("*") if path.is_file())


`torch_dtype` is deprecated! Use `dtype` instead!


C:\Users\USER\AppData\Roaming\Python\Python314\site-packages\transformers\models\t5\modeling_t5.py:1271: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  if sequence_length != 1:
C:\Users\USER\AppData\Roaming\Python\Python314\site-packages\transformers\cache_utils.py:132: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  if not self.is_initialized or self.keys.numel() == 0:
Could not find ONNX initializer for torch parameter decoder.embed_tokens.weight. decoder.embed_tokens.weight will not be checked for deduplication.
Could not find ONNX initializer for torc

['config.json',
 'generation_config.json',
 'onnx/config.json',
 'onnx/decoder_model.onnx',
 'onnx/decoder_model_merged.onnx',
 'onnx/decoder_with_past_model.onnx',
 'onnx/encoder_model.onnx',
 'onnx/generation_config.json',
 'onnx/special_tokens_map.json',
 'onnx/spiece.model',
 'onnx/tokenizer.json',
 'onnx/tokenizer_config.json',
 'special_tokens_map.json',
 'spiece.model',
 'tokenizer.json',
 'tokenizer_config.json']

## Use in Next.js

After the export cell completes, keep the generated `public/models/resume-refiner/` directory in the application deployment. The app loads it with `@huggingface/transformers`, disables remote model downloads, and sends only parser-produced skills, experience, and projects to it. The generated ONNX files can be large; use a host and deployment workflow that supports the resulting artifact size.